In [1]:
#%pip install azure-identity azure-storage-blob pandas
#%pip install scikit-learn seaborn matplotlib
#%pip install sqlalchemy pyodbc

In [2]:
import urllib.request
ip = urllib.request.urlopen('https://ident.me').read().decode('utf8')
print(f"My Compute IP is: {ip}")

My Compute IP is: 4.193.227.133


## Load Unstructed Data from Azure Blob Storage

In [3]:
import pandas as pd
import json
from io import BytesIO
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient

from datetime import datetime

# --- 1. ENTERPRISE CONFIGURATION ---
ACCOUNT_URL = "https://rawtradingdata26.blob.core.windows.net"
CONTAINER_NAME = "raw-market-data"

# Dynamically generate today's date string (e.g., '20260401')
today_str = datetime.now().strftime('%Y%m%d')

# Inject the date into the filenames
MARKET_BLOB_NAME = f"market_data_{today_str}.csv"
MACRO_BLOB_NAME = f"macro_data_{today_str}.json"

# Authenticate securely using the workspace's Managed Identity
print("Authenticating with Azure Managed Identity...")
credential = DefaultAzureCredential()
blob_service_client = BlobServiceClient(account_url=ACCOUNT_URL, credential=credential)
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

# --- 2. LOAD MARKET DATA (CSV) ---
print("Downloading Market Data securely...")
market_blob_client = container_client.get_blob_client(MARKET_BLOB_NAME)
market_download = market_blob_client.download_blob().readall()

df_market = pd.read_csv(BytesIO(market_download), header=[0, 1], index_col=0, parse_dates=True)
print(f"Market Data Loaded: {df_market.shape}")

# --- 3. LOAD MACRO DATA (JSON) ---
print("Downloading Macro Data securely...")
macro_blob_client = container_client.get_blob_client(MACRO_BLOB_NAME)
macro_download = macro_blob_client.download_blob().readall()

macro_json = json.loads(macro_download)
df_macro = pd.DataFrame(macro_json['observations'])

df_macro['date'] = pd.to_datetime(df_macro['date'])
df_macro['value'] = pd.to_numeric(df_macro['value'], errors='coerce')
df_macro = df_macro[['date', 'value']].rename(columns={'value': 'CPI'}).set_index('date')
print(f"Macro Data Loaded: {df_macro.shape}")

# Verify the data
print(df_market.head())

Authenticating with Azure Managed Identity...


ResourceNotFoundError: The specified blob does not exist.
RequestId:0f00a13e-401e-0067-70d1-c1843b000000
Time:2026-04-01T12:13:53.2152058Z
ErrorCode:BlobNotFound
Content: <?xml version="1.0" encoding="utf-8"?><Error><Code>BlobNotFound</Code><Message>The specified blob does not exist.
RequestId:0f00a13e-401e-0067-70d1-c1843b000000
Time:2026-04-01T12:13:53.2152058Z</Message></Error>

## Clean, Align, Feature Engineering, Scaling and Clustering

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. CLEAN & ALIGN ---
if isinstance(df_market.columns, pd.MultiIndex):
    df_market.columns = [f"{col[0]}_{col[1]}" for col in df_market.columns.values]

df_market = df_market.ffill()

if df_market.index.tz is not None:
    df_market.index = df_market.index.tz_localize(None)

df_macro_daily = df_macro.resample('D').ffill()

# Now perform the join safely
df_merged = df_market.join(df_macro_daily, how='left')

# THE FIX: Drag the last known CPI value forward to today to fix the Government Reporting Lag!
df_merged['CPI'] = df_merged['CPI'].ffill()

# --- 2. FEATURE ENGINEERING ---
df_merged['SPY_Daily_Return'] = df_merged['Close_SPY'].pct_change()
df_merged['SPY_Volatility_20d'] = df_merged['SPY_Daily_Return'].rolling(window=20).std()

# Only drop rows if our critical Machine Learning features are missing!
features = ['SPY_Daily_Return', 'SPY_Volatility_20d', 'CPI']
df_cleaned = df_merged.dropna(subset=features).copy()

print(f"Cleaned ML Dataset Shape: {df_cleaned.shape}")

# --- 3. SCALING & CLUSTERING ---
features = ['SPY_Daily_Return', 'SPY_Volatility_20d', 'CPI']
X = df_cleaned[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_cleaned['Regime'] = kmeans.fit_predict(X_scaled).astype(str)

# --- 4. VISUALIZE ---
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_cleaned, x='CPI', y='SPY_Volatility_20d', hue='Regime', palette='viridis', s=100, alpha=0.8)
plt.title('Cloud-Computed Market Regimes')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print("\n--- Cloud Cluster Averages ---")
print(df_cleaned.groupby('Regime')[features].mean())

## Upload Structured Data to Azure SQL Database

In [ ]:
import pandas as pd
import struct
import pyodbc
from sqlalchemy import create_engine
from azure.identity import DefaultAzureCredential

print("--- SAVING STRUCTURED DATA TO AZURE SQL (TOKEN AUTH) ---")

# 1. SQL Server Config
server = 'quant-server-123.database.windows.net' # UPDATE THIS
database = 'trading-db'                          # UPDATE THIS
driver = '{ODBC Driver 17 for SQL Server}'

try:
    # 2. Fetch the Access Token using the credential that worked for Blob Storage
    print("Getting Azure AD Access Token...")
    credential = DefaultAzureCredential()
    token_object = credential.get_token("https://database.windows.net/.default")
    
    # 3. Pack the token into a byte struct (Microsoft's required format for pyodbc)
    token_as_bytes = bytes(token_object.token, "UTF-8")
    encoded_token = token_as_bytes.decode("UTF-8").encode("UTF-16-LE")
    token_struct = struct.pack(f"<I{len(encoded_token)}s", len(encoded_token), encoded_token)

    # 4. Standard Connection String (Notice we removed the Authentication parameter)
    conn_str = f"DRIVER={driver};SERVER=tcp:{server},1433;DATABASE={database};Encrypt=yes;TrustServerCertificate=no;"

    # 5. Create a custom connection builder for SQLAlchemy to inject the token
    SQL_COPT_SS_ACCESS_TOKEN = 1256
    def get_conn():
        return pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct})

    # 6. Initialize SQLAlchemy engine
    engine = create_engine("mssql+pyodbc://", creator=get_conn, fast_executemany=True)

    # 7. Write the DataFrame to SQL
    df_sql = df_cleaned.reset_index()
    print("Writing feature-engineered data to SQL Database...")
    df_sql.to_sql('ProcessedMarketData', engine, if_exists='replace', index=False)
    
    print("✅ Successfully saved structured data to Azure SQL Database using Entra Token!")
    
    latest_regime = df_sql.iloc[-1]['Regime']
    print(f"\nToday's AI Classification: Regime {latest_regime}")
    
except Exception as e:
    print(f"❌ Failed to upload to SQL: {e}")

## Dynamic Back-test Code

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("--- EVALUATING COMPREHENSIVE SECTOR ROTATION STRATEGY ---")

# THE FIX: Drop any columns that completely failed to download from Yahoo Finance
df_cleaned = df_cleaned.dropna(axis=1, how='all')

# 1. Dynamically find all valid asset columns that survived
price_columns = [col for col in df_cleaned.columns if col.startswith('Close_')]
assets = [col.split('_')[1] for col in price_columns]

# 2. Calculate Forward Returns for ALL valid assets in a loop
return_columns = []
for asset in assets:
    col_name = f'{asset}_Fwd_Return'
    df_cleaned[col_name] = df_cleaned[f'Close_{asset}'].pct_change().shift(-1)
    return_columns.append(col_name)

# THE FIX: Only drop rows where the *Returns* are missing (safely drops the final unknown day)
backtest_df = df_cleaned.dropna(subset=return_columns)

# 3. Group by Regime and calculate annualized returns
regime_performance = backtest_df.groupby('Regime')[return_columns].mean()
annualized_performance = regime_performance * 252 * 100 

# Clean up column names for the chart
annualized_performance.columns = assets

print("\n📈 Annualized Expected Return by Regime (%)")
display(annualized_performance.round(2))

# 4. Visualize the Comprehensive Rotation
plt.figure(figsize=(14, 7))
annualized_performance.plot(kind='bar', figsize=(14, 7), colormap='tab10', edgecolor='black')

plt.title('Strategy Evaluation: Comprehensive Sector Performance by AI Market Regime', fontsize=14, fontweight='bold')
plt.ylabel('Annualized Expected Return (%)', fontsize=12)
plt.xlabel('AI Identified Regime', fontsize=12)
plt.axhline(0, color='black', linewidth=1.5)
plt.legend(title='Sectors / Baseline', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout() 
plt.show()

## Exploratory Data Analysis (EDA)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("--- GENERATING ACADEMIC EDA & FEATURE ENGINEERING VISUALS ---\n")

# --- 1. ADVANCED FEATURE ENGINEERING (RSI, MACD, Moving Averages) ---
# We use SPY (The broader market) to demonstrate the technical indicators
df_eda = df_cleaned.copy()

# Calculate 50-Day and 200-Day Simple Moving Averages
df_eda['SPY_SMA_50'] = df_eda['Close_SPY'].rolling(window=50).mean()
df_eda['SPY_SMA_200'] = df_eda['Close_SPY'].rolling(window=200).mean()

# Calculate MACD (Moving Average Convergence Divergence)
exp1 = df_eda['Close_SPY'].ewm(span=12, adjust=False).mean()
exp2 = df_eda['Close_SPY'].ewm(span=26, adjust=False).mean()
df_eda['MACD'] = exp1 - exp2
df_eda['MACD_Signal'] = df_eda['MACD'].ewm(span=9, adjust=False).mean()

# Calculate RSI (Relative Strength Index - 14 Day)
delta = df_eda['Close_SPY'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
df_eda['RSI_14'] = 100 - (100 / (1 + rs))

# --- 2. THE CORRELATION MATRIX HEATMAP ---
print("Generating Sector Correlation Heatmap...")
plt.figure(figsize=(10, 8))
# Dynamically grab whatever sectors successfully downloaded
price_cols = [col for col in df_eda.columns if col.startswith('Close_')]
clean_labels = [col.split('_')[1] for col in price_cols]

corr_matrix = df_eda[price_cols].corr()
corr_matrix.columns = clean_labels
corr_matrix.index = clean_labels

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5, fmt=".2f")
plt.title("S&P 500 Sector Correlation Matrix (10-Year)", fontsize=14, fontweight='bold')
plt.show()

# --- 3. TIME-SERIES VISUALIZATION & SIGNAL HIGHLIGHTING ---
print("\nGenerating Time-Series Technical Analysis Chart...")
# We will plot the last 2 years (500 trading days) so the chart is readable
df_plot = df_eda.tail(500) 

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12), gridspec_kw={'height_ratios': [3, 1, 1]})
fig.suptitle('Agentic Workflow EDA: Market Patterns & Risk Indicators (SPY)', fontsize=16, fontweight='bold')

# Subplot 1: Price and Moving Averages
ax1.plot(df_plot.index, df_plot['Close_SPY'], label='SPY Close Price', color='black', linewidth=1.5)
ax1.plot(df_plot.index, df_plot['SPY_SMA_50'], label='50-Day MA (Short Term)', color='blue', linestyle='--')
ax1.plot(df_plot.index, df_plot['SPY_SMA_200'], label='200-Day MA (Long Term)', color='red', linestyle='--')
ax1.set_ylabel('Price (USD)')
ax1.legend(loc='upper left')
ax1.grid(alpha=0.3)

# Subplot 2: MACD
ax2.plot(df_plot.index, df_plot['MACD'], label='MACD', color='purple')
ax2.plot(df_plot.index, df_plot['MACD_Signal'], label='Signal Line', color='orange')
ax2.axhline(0, color='black', linestyle='--', linewidth=1)
ax2.set_ylabel('MACD')
ax2.legend(loc='upper left')
ax2.grid(alpha=0.3)

# Subplot 3: RSI
ax3.plot(df_plot.index, df_plot['RSI_14'], label='RSI (14-Day)', color='green')
ax3.axhline(70, color='red', linestyle='--', alpha=0.5, label='Overbought (70)')
ax3.axhline(30, color='blue', linestyle='--', alpha=0.5, label='Oversold (30)')
ax3.set_ylabel('RSI')
ax3.set_xlabel('Date')
ax3.legend(loc='upper left')
ax3.grid(alpha=0.3)

plt.tight_layout()
fig.subplots_adjust(top=0.95)
plt.show()